In [1]:

# MountainCar-v0 Environment Setup
# CS4287 Assignment 2 - Deep Reinforcement Learning

!pip install -q "gymnasium[classic-control]" #install GYMNASIUM inorder to use the mountain car environment


MountainCar-v0 Environment
We selected MountainCar-v0 from the Gymnasium Classic Control environments for this assignment.

The car positioned between two hills thats what the environment contains. The main goal is to reach the flag at the top of the right hill.Also the car's engine is not powerful enough to drive directly to the goal.The agent needs to learn how to build momentum by moving back and forth.

Two observations are provided by the environment and these are the car's position and velocity. The agent wil be able to choose between three actions which are push left (0), no push (1), or push right (2).

The agent also receives a reward of -1 for each step. The reward encourages it to reach the goal in as few steps as possible. When the car reaches the goal or reaches the maximum of 200 steps thats when an episode ends.

We chose this environment because the agent must learn a sequence of actions through trial and error. Instead of it being given the correct action for every situation.

In [2]:

import gymnasium as gym

# Create the mountaincar environment
env = gym.make("MountainCar-v0")

# environment reset new episode
state, info = env.reset(seed=42)

# show information about the environment
print("Environment:", env.spec.id)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial state:", state)

env.close()


Environment: MountainCar-v0
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Action space: Discrete(3)
Initial state: [-0.4452088  0.       ]


In [3]:

import gymnasium as gym

# Create a new MountainCar environment
env = gym.make("MountainCar-v0")

# Start a new episode
state, info = env.reset(seed=42)

# move the car to the right
action = 2

# Apply the action to the environment
next_state, reward, terminated, truncated, info = env.step(action)

# Display the results
print("Action taken:", action)
print("New position:", next_state[0])
print("New velocity:", next_state[1])
print("Reward received:", reward)
print("Episode finished:", terminated or truncated)

env.close()


Action taken: 2
New position: -0.44479132
New velocity: 0.00041747934
Reward received: -1.0
Episode finished: False


Data Collection

I collected data by allowing the agent to take 10 random actions in the MountainCar environment.I recorded the currebnt state,the action taken ,the reward,the next state and whether the episode had finished after each action.I stored these experience because they could be used for later when training the DQN model.10 actions were only used to test that our data collection code was working correctly.





In [4]:

import gymnasium as gym

# Create the MountainCar environment
env = gym.make("MountainCar-v0")

# Start a new episode
state, info = env.reset(seed=42)

# Store the experiences collected
experiences = []

# Let the car take 10 random actions
for step in range(10):

    # Choose a random action (0, 1, or 2)
    action = env.action_space.sample()

    # Apply the action
    next_state, reward, terminated, truncated, info = env.step(action)

    # Check if the episode has finished
    done = terminated or truncated

    # Save the experience
    experience = (state.copy(), action, reward, next_state.copy(), done)
    experiences.append(experience)

    # Display information
    print(f"Step {step + 1}: Action={action}, Reward={reward}, Done={done}")

    # Update the state for the next step
    state = next_state

    # Stop if the episode finishes
    if done:
        break

# Display the number of collected experiences
print("\nTotal experiences collected:", len(experiences))

# Close the environment
env.close()


Step 1: Action=1, Reward=-1.0, Done=False
Step 2: Action=2, Reward=-1.0, Done=False
Step 3: Action=0, Reward=-1.0, Done=False
Step 4: Action=1, Reward=-1.0, Done=False
Step 5: Action=1, Reward=-1.0, Done=False
Step 6: Action=2, Reward=-1.0, Done=False
Step 7: Action=1, Reward=-1.0, Done=False
Step 8: Action=2, Reward=-1.0, Done=False
Step 9: Action=0, Reward=-1.0, Done=False
Step 10: Action=2, Reward=-1.0, Done=False

Total experiences collected: 10


Experience Replay Buffer

We used an experience replay buffer in order to store the agent's previous experiences. Each experience has the current state, action, reward, next state and whether the episode has finished.

The buffer uses a deque with a maximum capacity of 10,000 experiences. Older experiences are removed automatically when the buffer becomes full.

In order to select experiences for training, we used random sampling. It helps to reduce the relationship between consecutive experiences and makes learning more stable.

In [5]:

import random
from collections import deque

# Create a memory buffer to store experiences
class ReplayBuffer:

    def __init__(self, capacity=10000):
        # Keep only the most recent experiences
        self.memory = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        # Store one experience in memory
        self.memory.append(
            (state, action, reward, next_state, done)
        )

    def sample(self, batch_size):
        # Randomly select experiences for training
        return random.sample(self.memory, batch_size)

    def __len__(self):
        # Return how many experiences are stored
        return len(self.memory)


# Create the replay buffer
replay_buffer = ReplayBuffer(capacity=10000)

# Add the 10 experiences we collected earlier
for experience in experiences:
    replay_buffer.add(*experience)

# Randomly select 4 experiences from memory
batch = replay_buffer.sample(batch_size=4)

print("Experiences stored:", len(replay_buffer))
print("Experiences sampled:", len(batch))


Experiences stored: 10
Experiences sampled: 4


DQN Network Architecture

To create the Deep Q-Network (DQN) for the MountainCar environment I used the pyTorch.
however the network has two inputs which are the car's position and velocity.The network also has two hidden layer with 64 neurons each.These hidden layers help the network learn patterns from the informatio it receives.I also used the ReLU activationn function between the hidden layers. The output has 3 neurons because the agent has the possible actions which are push left, no push and push right.Q-value for each action is predicted by the network.The values help the agent to make a decision on which action to take.However the network still needs training before it can make useful decisions.






In [6]:

import torch
import torch.nn as nn

# Define the Deep Q-Network
class DQN(nn.Module):

    def __init__(self, state_size=2, action_size=3):
        super(DQN, self).__init__()

        # Create the neural network layers
        self.network = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )

    def forward(self, state):
        # Pass the state through the network
        return self.network(state)


# Create the DQN model
model = DQN()

# Display the network architecture
print(model)

# Test the network using a sample state
sample_state = torch.tensor(
    [-0.5, 0.0], dtype=torch.float32
)

q_values = model(sample_state)

print("\nPredicted Q-values:", q_values.detach().numpy())
print("Recommended action:", torch.argmax(q_values).item())


DQN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=3, bias=True)
  )
)

Predicted Q-values: [0.08873133 0.10429733 0.02317545]
Recommended action: 1


Testing and Intergration

In this section I tested the following things to see if they are working together, the MouintainCar environment,DQN network and replay buffer.To predict the Q value and to chose an action I used DQN.I also checked if the the reply buffer could randomly select for experiences.This result showed that all different parts were working correctly.Although the DQN has not yet been trained.

In [7]:

import torch

# Create a new MountainCar environment
test_env = gym.make("MountainCar-v0")

# Start an episode
state, info = test_env.reset(seed=42)

# Convert the state into a PyTorch tensor
state_tensor = torch.tensor(state, dtype=torch.float32)

# Use the DQN to predict Q-values
with torch.no_grad():
    q_values = model(state_tensor)

# Choose the action with the highest Q-value
action = torch.argmax(q_values).item()

# Apply the selected action
next_state, reward, terminated, truncated, info = test_env.step(action)

# Test the replay buffer by sampling 4 experiences
sample_batch = replay_buffer.sample(batch_size=4)

# Display the results
print("State shape:", state_tensor.shape)
print("Q-values shape:", q_values.shape)
print("Selected action:", action)
print("Reward:", reward)
print("Replay buffer size:", len(replay_buffer))
print("Sample batch size:", len(sample_batch))

test_env.close()


State shape: torch.Size([2])
Q-values shape: torch.Size([3])
Selected action: 1
Reward: -1.0
Replay buffer size: 10
Sample batch size: 4
